# ANN for fashion mnist pytorch GPU Optimized

to reduce overfitting, we are using `Dropouts`
<br>
**Dropout**
- Applied to hidden layers
- Applied after the ReLU activation functions.
- Randoml turns off p% neurons in hidden layer during each forward pass.
- Has a regularization effect
- During evaluaion dropout is used.

In [16]:
import kagglehub
path = kagglehub.dataset_download("zalando-research/fashionmnist")
print(path)

Using Colab cache for faster access to the 'fashionmnist' dataset.
/kaggle/input/fashionmnist


In [17]:
import os

for root,dirs,files in os.walk(path):
  for file in files:
    print(os.path.join(root,file))

/kaggle/input/fashionmnist/t10k-labels-idx1-ubyte
/kaggle/input/fashionmnist/t10k-images-idx3-ubyte
/kaggle/input/fashionmnist/fashion-mnist_test.csv
/kaggle/input/fashionmnist/fashion-mnist_train.csv
/kaggle/input/fashionmnist/train-labels-idx1-ubyte
/kaggle/input/fashionmnist/train-images-idx3-ubyte


In [18]:
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [19]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using {device}")

using cuda


In [20]:
train_df = pd.read_csv(os.path.join(path, "fashion-mnist_train.csv"))
test_df=pd.read_csv(os.path.join(path,"fashion-mnist_test.csv"))
X_train=train_df.drop(columns=["label"]).values
y_train=train_df["label"].values

X_test=test_df.drop(columns=["label"]).values
y_test=test_df["label"].values

X_train,X_val,y_train,y_val=train_test_split(X_train,y_train,test_size=0.2,random_state=42,stratify=y_train)

In [21]:
# Normalize
X_train = X_train.astype(np.float32) / 255.0
X_val = X_val.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)

X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.long)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

In [22]:
from torch.utils.data import TensorDataset
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    pin_memory=True  # speed of training over GPU increases
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    pin_memory=True
)

In [23]:
import torch.nn as nn
class MyNN(nn.Module):

    def __init__(self, num_features):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(num_features,128),
            nn.BatchNorm1d(128),  #our dataset is 1d  (batch normalization)
            nn.ReLU(),
            nn.Dropout(p=0.3),  # added dropout functions, 0.3 =30% dropouts
            nn.Linear(128,64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(64,10)
        )

    def forward(self,x):
        return self.model(x)

  # model and optimizer

model = MyNN(X_train.shape[1]).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1,
    weight_decay=1e-4,   #regulaization coefficient
)

In [24]:
epochs = 100

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for batch_features, batch_labels in train_loader:

        # Move batch to GPU
        batch_features = batch_features.to(device, non_blocking=True)
        batch_labels = batch_labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(batch_features)

        loss = criterion(outputs, batch_labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch [{epoch+1}/{epochs}] Loss: {avg_loss:.4f}")

Epoch [1/100] Loss: 0.6566
Epoch [2/100] Loss: 0.4841
Epoch [3/100] Loss: 0.4415
Epoch [4/100] Loss: 0.4166
Epoch [5/100] Loss: 0.4004
Epoch [6/100] Loss: 0.3850
Epoch [7/100] Loss: 0.3672
Epoch [8/100] Loss: 0.3561
Epoch [9/100] Loss: 0.3534
Epoch [10/100] Loss: 0.3493
Epoch [11/100] Loss: 0.3357
Epoch [12/100] Loss: 0.3280
Epoch [13/100] Loss: 0.3211
Epoch [14/100] Loss: 0.3176
Epoch [15/100] Loss: 0.3150
Epoch [16/100] Loss: 0.3109
Epoch [17/100] Loss: 0.3040
Epoch [18/100] Loss: 0.2984
Epoch [19/100] Loss: 0.2945
Epoch [20/100] Loss: 0.2904
Epoch [21/100] Loss: 0.2876
Epoch [22/100] Loss: 0.2830
Epoch [23/100] Loss: 0.2808
Epoch [24/100] Loss: 0.2824
Epoch [25/100] Loss: 0.2704
Epoch [26/100] Loss: 0.2703
Epoch [27/100] Loss: 0.2666
Epoch [28/100] Loss: 0.2643
Epoch [29/100] Loss: 0.2624
Epoch [30/100] Loss: 0.2615
Epoch [31/100] Loss: 0.2608
Epoch [32/100] Loss: 0.2541
Epoch [33/100] Loss: 0.2550
Epoch [34/100] Loss: 0.2482
Epoch [35/100] Loss: 0.2455
Epoch [36/100] Loss: 0.2464
E

In [26]:
# set model to eval mode
model.eval()


# evalutaion code to do evaluation

total=0
correct=0

with torch.no_grad():
    for batch_features,batch_labels in train_loader:
        batch_features = batch_features.to(device, non_blocking=True)
        batch_labels = batch_labels.to(device, non_blocking=True)
        outputs=model(batch_features)

        _,predicted=torch.max(outputs,1)
        total=total+batch_labels.shape[0]

        correct=correct+(predicted==batch_labels).sum().item()


print(correct/total)

0.9602916666666667


In [25]:
# set model to eval mode
model.eval()


# evalutaion code to do evaluation

total=0
correct=0

with torch.no_grad():
    for batch_features,batch_labels in val_loader:
        batch_features = batch_features.to(device, non_blocking=True)
        batch_labels = batch_labels.to(device, non_blocking=True)
        outputs=model(batch_features)

        _,predicted=torch.max(outputs,1)
        total=total+batch_labels.shape[0]

        correct=correct+(predicted==batch_labels).sum().item()


print(correct/total)

0.8924166666666666
